# BNPL Governance Workshop - Part 2: Flink Stream Processing

This notebook picks up where `kredivo-01.ipynb` left off. Here we use **Confluent Cloud Flink SQL** to:

1. Produce a stream of orders that deliberately contains duplicate records (simulating retries / at-least-once delivery).
2. Run a Flink SQL stream that continuously calculates **what % of the incoming records are duplicates**.
3. Deduplicate the stream (keep only the first occurrence of each `order_id`) into a clean topic.
4. **Filter** the stream (only high-value confirmed orders) and **trim** it down to a smaller set of fields, into a new topic.
5. **Enrich** the stream by joining it against a small product reference table, into a new topic.

**Requirements:**
- `.env` file in this directory with your Confluent Cloud credentials (same as `kredivo-01.ipynb`).
- For the Flink cells to actually execute (not just print the SQL), you need the [Confluent CLI](https://docs.confluent.io/confluent-cli/current/overview.html) installed **and logged in** (`confluent login`), with `FLINK_COMPUTE_POOL_ID`, `ENV_ID`, and `CONFLUENT_CLUSTER_ID` set in `.env`. If the CLI isn't available/logged in, every Flink cell below prints the SQL it would run and explains what to expect instead of failing.


### Install dependencies

Installs the Kafka client, `.env` loader, and pandas (used later to display Flink query results as tables).

**Example:** after this runs, `import pandas as pd` and `from confluent_kafka import Producer` succeed in the next cell.

In [58]:
RUN_IN_NOTEBOOK = True  # Set to False if you'd rather copy PIP_CMD and run it yourself in a terminal (with .venv activated)

PIP_CMD = "pip install -q confluent-kafka python-dotenv pandas"

if RUN_IN_NOTEBOOK:
    !{PIP_CMD}
else:
    print(f"Skipping -- run this yourself in a terminal (with .venv activated):\n\n  $ {PIP_CMD}")

### Load credentials and build per-participant topic names

Reads Kafka, Schema Registry, and Flink REST credentials from `.env`, then derives a unique `PARTICIPANT` suffix so every topic (`orders_source_<you>`, `orders_deduped_<you>`, etc.) is namespaced to you and won't collide with other workshop attendees running the same notebook.

**Example:** if `PARTICIPANT_ID` isn't set in `.env`, it falls back to your OS username — e.g. `TOPIC_ORDERS` becomes `orders_source_ahartono`.

In [59]:
import os
import json as pyjson
import time
import uuid
import random
import subprocess
import requests
import datetime as dt
from dotenv import load_dotenv
import pandas as pd

from confluent_kafka import Producer, Consumer
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka.schema_registry import SchemaRegistryClient, Schema
from confluent_kafka.schema_registry.avro import AvroSerializer, AvroDeserializer
from confluent_kafka.serialization import StringSerializer, SerializationContext, MessageField

random.seed(7)
OK, FAIL, WARN, INFO = "✅", "❌", "⚠️", "ℹ️"

load_dotenv(".env")

bootstrap_servers = os.getenv("CCLOUD_BOOTSTRAP_SERVERS")
kafka_api_key = os.getenv("CCLOUD_API_KEY")
kafka_api_secret = os.getenv("CCLOUD_API_SECRET")
sr_url = os.getenv("SCHEMA_REGISTRY_URL")
sr_api_key = os.getenv("SCHEMA_REGISTRY_API_KEY")
sr_api_secret = os.getenv("SCHEMA_REGISTRY_API_SECRET")

FLINK_POOL = os.getenv("FLINK_COMPUTE_POOL_ID", "")

# Flink SQL REST API credentials -- a *separate* API key, scoped to Flink, distinct
# from the Kafka and Schema Registry keys above. Created via Cloud Console ->
# environment -> Flink -> API keys (or `confluent flink region list` + `api-key create
# --resource <flink-region-id>`).
FLINK_REST_REGION = os.getenv("CLOUD_REGION", "").lower()
FLINK_REST_CLOUD = os.getenv("CLOUD_PROVIDER", "").lower()
FLINK_ORG_ID = os.getenv("ORG_ID", "")
FLINK_ENV_ID = os.getenv("ENV_ID", "")
FLINK_CLUSTER_ID = os.getenv("CONFLUENT_CLUSTER_ID", "")
FLINK_API_KEY = os.getenv("FLINK_API_KEY", "")
FLINK_API_SECRET = os.getenv("FLINK_API_SECRET", "")

FLINK_REST_AVAILABLE = all([
    FLINK_REST_REGION, FLINK_REST_CLOUD, FLINK_ORG_ID, FLINK_ENV_ID, FLINK_CLUSTER_ID,
    FLINK_API_KEY, FLINK_API_SECRET, FLINK_POOL,
])

PARTICIPANT = os.getenv("PARTICIPANT_ID") or os.getenv("USER") or "p00"
PARTICIPANT = "".join(c for c in PARTICIPANT if c.isalnum() or c in "-_")[:20] or "p00"

kafka_conf = {
    "bootstrap.servers": bootstrap_servers,
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "PLAIN",
    "sasl.username": kafka_api_key,
    "sasl.password": kafka_api_secret,
}

sr_conf = {
    "url": sr_url,
    "basic.auth.user.info": f"{sr_api_key}:{sr_api_secret}",
}
sr_client = SchemaRegistryClient(sr_conf)

TOPIC_ORDERS = f"orders_source_{PARTICIPANT}"
TOPIC_DUP_STATS = f"orders_dup_stats_{PARTICIPANT}"
TOPIC_DEDUPED = f"orders_deduped_{PARTICIPANT}"
TOPIC_FILTERED = f"orders_high_value_{PARTICIPANT}"
TOPIC_CATALOG = f"product_catalog_{PARTICIPANT}"
TOPIC_ENRICHED = f"orders_enriched_{PARTICIPANT}"

admin = AdminClient(kafka_conf)

def create_topic(name, num_partitions=1, replication_factor=3):
    """Confluent Cloud clusters have auto topic creation disabled by default,
    so every topic used below must be created explicitly first."""
    fs = admin.create_topics([NewTopic(name, num_partitions=num_partitions, replication_factor=replication_factor)])
    for topic, f in fs.items():
        try:
            f.result()
            print(f"{OK} Created topic: {topic}")
        except Exception as e:
            print(f"{INFO} Topic note for {topic}: {e}")

# Only pre-create the topics that Python producers write into directly. The other four
# (dup-stats, deduped, filtered, enriched) are owned end-to-end by Flink -- their own
# `CREATE TABLE` DDL in later cells provisions the topic *and* its Schema Registry schema
# together. Pre-creating a topic before Flink defines its schema backfires: Confluent Cloud
# auto-infers a schemaless `key: BYTES, val: BYTES` table for it immediately, and `CREATE
# TABLE IF NOT EXISTS` then silently keeps that (wrong) inferred table instead of the one
# Flink's DDL describes.
for t in (TOPIC_ORDERS, TOPIC_CATALOG):
    create_topic(t)

print(f"\n{INFO} Participant namespace : {PARTICIPANT}")
print(f"{OK if FLINK_REST_AVAILABLE else WARN} Flink REST credentials available: {FLINK_REST_AVAILABLE}"
      + ("" if FLINK_REST_AVAILABLE else " -- Flink cells below will explain-only until "
         "CLOUD_REGION, CLOUD_PROVIDER, ORG_ID, ENV_ID, FLINK_API_KEY, FLINK_API_SECRET are set in .env."))


%6|1786891989.874|GETSUBSCRIPTIONS|rdkafka#producer-34| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to WEr6jFpuSbOlJ8X9dVsVJA


✅ Created topic: orders_source_ahartono
✅ Created topic: product_catalog_ahartono

ℹ️ Participant namespace : ahartono
✅ Flink REST credentials available: True


## Step 1 -- Register the `Order` schema and create the source topic

A simple Avro contract for an order event. `channel` is included deliberately -- it's an
internal routing field (`WEB` / `APP` / `STORE`) that downstream analytics consumers don't need.
We'll strip it out later in the filter+trim step.


In [60]:
ORDER_SCHEMA = {
    "type": "record",
    "name": "Order",
    "namespace": "com.bnpl.orders",
    "doc": "A single order event, as emitted by the checkout service.",
    "fields": [
        {"name": "order_id",    "type": "string", "doc": "Business key; may arrive more than once (retries)"},
        {"name": "customer_id", "type": "string"},
        {"name": "product_id",  "type": "string"},
        {"name": "quantity",    "type": "int"},
        {"name": "amount",      "type": "double"},
        {"name": "order_ts",    "type": {"type": "long", "logicalType": "timestamp-millis"}},
        {"name": "status",      "type": "string", "doc": "PENDING | CONFIRMED | CANCELLED"},
        {"name": "channel",     "type": "string", "doc": "WEB | APP | STORE -- internal routing only"},
    ],
}

orders_subject = f"{TOPIC_ORDERS}-value"
order_schema_id = sr_client.register_schema(orders_subject, Schema(pyjson.dumps(ORDER_SCHEMA), "AVRO"))
print(f"{OK} Registered '{orders_subject}' -> schema id {order_schema_id}")

order_ser = AvroSerializer(sr_client, pyjson.dumps(ORDER_SCHEMA))
key_ser = StringSerializer("utf_8")
orders_producer = Producer(kafka_conf)


✅ Registered 'orders_source_ahartono-value' -> schema id 100031


## Step 2 -- Produce orders, deliberately including duplicates

We generate a batch of unique orders, then **resend a random subset of them 1-2 extra times**,
byte-for-byte identical -- exactly what happens when a producer retries after a slow ack, or an
upstream system at-least-once-delivers the same event twice.

We keep the exact ground truth (`GROUND_TRUTH_DUP_PCT`) so we can later check Flink's own
calculation against it.


In [ ]:
STATUSES = ["CONFIRMED", "CONFIRMED", "CONFIRMED", "PENDING", "CANCELLED"]
CHANNELS = ["WEB", "APP", "STORE"]
PRODUCT_IDS = [f"SKU-{i:03d}" for i in range(1, 11)]

N_UNIQUE_ORDERS = 2000
DUP_PROBABILITY = 0.40    # fraction of orders that get resent
MAX_EXTRA_SENDS = 2        # how many extra times a "duplicated" order gets resent

def make_order(order_id=None):
    return {
        "order_id":    order_id or str(uuid.uuid4()),
        "customer_id": f"CUST-{random.randint(1, 50):04d}",
        "product_id":  random.choice(PRODUCT_IDS),
        "quantity":    random.randint(1, 5),
        "amount":      round(random.uniform(20.0, 1200.0), 2),
        "order_ts":    dt.datetime.now(dt.timezone.utc),
        "status":      random.choice(STATUSES),
        "channel":     random.choice(CHANNELS),
    }

delivered = {"ok": 0, "err": []}
def on_delivery(err, msg):
    if err: delivered["err"].append(str(err))
    else:   delivered["ok"] += 1

total_sent = 0
duplicate_sends = 0

for _ in range(N_UNIQUE_ORDERS):
    order = make_order()
    ctx = SerializationContext(TOPIC_ORDERS, MessageField.VALUE)

    # first (original) send
    orders_producer.produce(
        topic=TOPIC_ORDERS,
        key=key_ser(order["order_id"]),
        value=order_ser(order, ctx),
        on_delivery=on_delivery,
    )
    total_sent += 1

    # maybe resend the *exact same* record a few more times
    if random.random() < DUP_PROBABILITY:
        extra = random.randint(1, MAX_EXTRA_SENDS)
        for _ in range(extra):
            orders_producer.produce(
                topic=TOPIC_ORDERS,
                key=key_ser(order["order_id"]),
                value=order_ser(order, ctx),
                on_delivery=on_delivery,
            )
            total_sent += 1
            duplicate_sends += 1

orders_producer.flush(30)

GROUND_TRUTH_TOTAL = total_sent
GROUND_TRUTH_UNIQUE = N_UNIQUE_ORDERS
GROUND_TRUTH_DUPLICATES = duplicate_sends
GROUND_TRUTH_DUP_PCT = round(duplicate_sends / total_sent * 100, 2)

print(f"{OK} Delivered {delivered['ok']}/{total_sent} records to {TOPIC_ORDERS}")
if delivered["err"]:
    print(f"{FAIL} Delivery errors: {delivered['err'][:3]}")

print(f"\n{INFO} Ground truth (known because we generated the data):")
print(f"     unique orders     : {GROUND_TRUTH_UNIQUE}")
print(f"     total records sent: {GROUND_TRUTH_TOTAL}")
print(f"     duplicate records : {GROUND_TRUTH_DUPLICATES}")
print(f"     duplicate %       : {GROUND_TRUTH_DUP_PCT}%")


%6|1786891995.304|GETSUBSCRIPTIONS|rdkafka#producer-35| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to 3tmxBkekTQqcKPaUBi6/pA


✅ Delivered 32254/32254 records to orders_source_ahartono

ℹ️ Ground truth (known because we generated the data):
     unique orders     : 20000
     total records sent: 32254
     duplicate records : 12254
     duplicate %       : 37.99%


## Step 3 -- Flink SQL: what % of the stream is duplicated?

`orders_source_<you>` is auto-inferred as a Flink table the moment the topic exists -- no DDL
needed to read it. The query below groups records into 2-minute tumbling windows and computes,
per window:

```
duplicate_pct = (total_records - distinct order_ids) / total_records * 100
```

We write the result to its own sink table (`orders_dup_stats_<you>`) so we can read it back with a
plain consumer afterwards, the same way the result of any monitoring/alerting stream would be
consumed downstream.

**Why this cell replaces old statements instead of just stopping them first:** this is a
long-running streaming `INSERT INTO ... SELECT` -- it never reaches `COMPLETED`, it just stays
`RUNNING` forever. Re-running this cell without cleaning up the previous run's statement leaves
*multiple* duplicate-rate jobs running at once, all writing into the same sink table -- Step 3b
then reads back an unpredictable mix of rows from all of them.

The safe way to do that swap is **new-then-old, never old-then-new**: submit the new `dupstats`
statement first, confirm via the REST API that it actually reached `RUNNING`, and only *then* stop
the previous run's statement(s). If instead the old statement were stopped first and the new
submission then failed (a transient network blip, a REST timeout), you'd end up with **zero**
active jobs and Step 3b would wait forever for data that will never arrive -- exactly what
happened when this cell's REST call briefly failed. Submitting new-then-old means the worst case
under a flaky connection is a short-lived overlap of two jobs, which is harmless, instead of an
unnoticed gap with none.

In [62]:

FLINK_REST_BASE = (
    f"https://flink.{FLINK_REST_REGION}.{FLINK_REST_CLOUD}.confluent.cloud"
    f"/sql/v1/organizations/{FLINK_ORG_ID}/environments/{FLINK_ENV_ID}/statements"
)

def run_flink_sql(sql, name_prefix="stmt", wait_seconds=60):
    """Submit one Flink SQL statement via the Flink SQL REST API. Returns (ok, output).

    DDL and bounded SELECTs reach phase COMPLETED quickly. A streaming INSERT INTO ... SELECT
    never completes by design -- for those, reaching RUNNING (not FAILED/FAILING) is success;
    we don't block waiting for it to finish.
    """
    if not FLINK_REST_AVAILABLE:
        print(f"{WARN} Flink REST credentials unavailable -- printing the SQL instead of running it:\n")
        print(sql)
        return False, "Flink REST credentials unavailable"

    name = f"{name_prefix}-{PARTICIPANT}-{uuid.uuid4().hex[:6]}"
    auth = (FLINK_API_KEY, FLINK_API_SECRET)

    try:
        resp = requests.post(
            FLINK_REST_BASE,
            auth=auth,
            headers={"content-type": "application/json"},
            json={
                "name": name,
                "spec": {
                    "statement": sql,
                    "compute_pool_id": FLINK_POOL,
                    "properties": {
                        "sql.current-catalog": FLINK_ENV_ID,
                        "sql.current-database": FLINK_CLUSTER_ID,
                    },
                },
            },
            timeout=30,
        )
    except Exception as e:
        print(f"{FAIL} Flink REST request failed: {e}")
        return False, str(e)

    if resp.status_code not in (200, 201):
        print(f"{FAIL} Flink REST error ({resp.status_code}): {resp.text.strip()[:500]}")
        return False, resp.text

    status_url = f"{FLINK_REST_BASE}/{name}"
    deadline = time.time() + wait_seconds
    phase, detail = None, ""
    while time.time() < deadline:
        r = requests.get(status_url, auth=auth, timeout=15)
        if r.status_code != 200:
            print(f"{FAIL} Could not poll statement status ({r.status_code}): {r.text.strip()[:300]}")
            return False, r.text
        status = r.json().get("status", {})
        phase = status.get("phase")
        detail = status.get("detail", "")
        if phase in ("RUNNING", "COMPLETED"):
            return True, f"{phase}: statement '{name}'"
        if phase in ("FAILED", "FAILING"):
            print(f"{FAIL} Statement '{name}' {phase}: {detail}")
            return False, detail
        time.sleep(3)

    ok = phase in ("RUNNING", "COMPLETED")
    if not ok:
        print(f"{WARN} Statement '{name}' still in phase={phase} after {wait_seconds}s: {detail}")
    return ok, f"phase={phase} detail={detail}"


def find_running_statements(name_prefix, exclude_name=None):
    """List this participant's own statements whose name starts with name_prefix and are
    still RUNNING/PENDING, optionally excluding one name (the statement we just submitted)."""
    if not FLINK_REST_AVAILABLE:
        return []
    auth = (FLINK_API_KEY, FLINK_API_SECRET)
    try:
        r = requests.get(FLINK_REST_BASE, auth=auth, timeout=30)
        r.raise_for_status()
        return [
            s["name"] for s in r.json().get("data", [])
            if s["name"].startswith(f"{name_prefix}-{PARTICIPANT}-")
            and s["name"] != exclude_name
            and (s.get("status") or {}).get("phase") in ("RUNNING", "PENDING")
        ]
    except Exception as e:
        print(f"{WARN} Could not list previous statements: {e}")
        return []


def stop_statement(name):
    auth = (FLINK_API_KEY, FLINK_API_SECRET)
    try:
        rr = requests.delete(f"{FLINK_REST_BASE}/{name}", auth=auth, timeout=30)
        print(f"{OK if rr.status_code in (200, 202, 204) else FAIL} Stopped previous statement {name}")
    except Exception as e:
        print(f"{WARN} Could not stop statement {name}: {e}")


def run_flink_sql_replacing(sql, name_prefix, wait_seconds=60):
    """Submit a new streaming statement and only stop the previous run's statement(s)
    *after* confirming the new one actually reached RUNNING/COMPLETED.

    Stopping old statements before the new submission is confirmed is what leaves a gap
    with zero active jobs when the new submission fails (e.g. a transient network issue) --
    the old job is already gone but no replacement ever started. Checking new-then-old
    avoids that: worst case if the network is flaky is a brief overlap of two jobs, which
    is safe, versus silently ending up with none.
    """
    name = f"{name_prefix}-{PARTICIPANT}-{uuid.uuid4().hex[:6]}"
    auth = (FLINK_API_KEY, FLINK_API_SECRET)

    if not FLINK_REST_AVAILABLE:
        print(f"{WARN} Flink REST credentials unavailable -- printing the SQL instead of running it:\n")
        print(sql)
        return False, "Flink REST credentials unavailable"

    try:
        resp = requests.post(
            FLINK_REST_BASE,
            auth=auth,
            headers={"content-type": "application/json"},
            json={
                "name": name,
                "spec": {
                    "statement": sql,
                    "compute_pool_id": FLINK_POOL,
                    "properties": {
                        "sql.current-catalog": FLINK_ENV_ID,
                        "sql.current-database": FLINK_CLUSTER_ID,
                    },
                },
            },
            timeout=30,
        )
    except Exception as e:
        print(f"{FAIL} Flink REST request failed: {e} -- previous statement (if any) left running.")
        return False, str(e)

    if resp.status_code not in (200, 201):
        print(f"{FAIL} Flink REST error ({resp.status_code}): {resp.text.strip()[:500]} -- "
              f"previous statement (if any) left running.")
        return False, resp.text

    status_url = f"{FLINK_REST_BASE}/{name}"
    deadline = time.time() + wait_seconds
    phase, detail = None, ""
    while time.time() < deadline:
        try:
            r = requests.get(status_url, auth=auth, timeout=15)
        except Exception as e:
            print(f"{WARN} Could not poll new statement status: {e} -- previous statement (if any) left running.")
            return False, str(e)
        if r.status_code != 200:
            print(f"{FAIL} Could not poll statement status ({r.status_code}): {r.text.strip()[:300]} -- "
                  f"previous statement (if any) left running.")
            return False, r.text
        status = r.json().get("status", {})
        phase = status.get("phase")
        detail = status.get("detail", "")
        if phase in ("RUNNING", "COMPLETED"):
            break
        if phase in ("FAILED", "FAILING"):
            print(f"{FAIL} New statement '{name}' {phase}: {detail} -- previous statement (if any) left running.")
            return False, detail
        time.sleep(3)
    else:
        print(f"{WARN} New statement '{name}' still in phase={phase} after {wait_seconds}s: {detail} -- "
              f"previous statement (if any) left running.")
        return False, f"phase={phase} detail={detail}"

    # New statement confirmed RUNNING/COMPLETED -- now safe to stop the old one(s).
    for old_name in find_running_statements(name_prefix, exclude_name=name):
        stop_statement(old_name)

    return True, f"{phase}: statement '{name}'"


DDL_DUP_STATS = f"""
CREATE TABLE IF NOT EXISTS `{TOPIC_DUP_STATS}` (
  `window_start`       TIMESTAMP(3),
  `window_end`          TIMESTAMP(3),
  `total_records`      BIGINT,
  `unique_records`     BIGINT,
  `duplicate_records`  BIGINT,
  `duplicate_pct`      DOUBLE
) WITH (
  'changelog.mode' = 'append',
  'value.format'   = 'avro-registry'
);
"""

DUP_STATS_QUERY = f"""
INSERT INTO `{TOPIC_DUP_STATS}`
SELECT
  `window_start`,
  `window_end`,
  COUNT(*)                                  AS `total_records`,
  COUNT(DISTINCT `order_id`)                AS `unique_records`,
  COUNT(*) - COUNT(DISTINCT `order_id`)      AS `duplicate_records`,
  CAST(COUNT(*) - COUNT(DISTINCT `order_id`) AS DOUBLE) / COUNT(*) * 100 AS `duplicate_pct`
FROM TABLE(
  TUMBLE(TABLE `{TOPIC_ORDERS}`, DESCRIPTOR(`$rowtime`), INTERVAL '2' MINUTE)
)
GROUP BY `window_start`, `window_end`;
"""

print(DDL_DUP_STATS)
ok, out = run_flink_sql(DDL_DUP_STATS, "ddl-dupstats")
print(f"{OK} dup-stats sink ready." if ok else f"{INFO} DDL not executed (see above).")

# Submit the new "dupstats" streaming job and only stop the previous run's job once the
# new one is confirmed RUNNING -- see run_flink_sql_replacing()'s docstring for why.
print(DUP_STATS_QUERY)
ok, out = run_flink_sql_replacing(DUP_STATS_QUERY, "dupstats")
print(f"{OK} Duplicate-rate stream submitted -- it stays RUNNING and keeps emitting." if ok
      else f"{INFO} Statement not submitted (see above). This is a long-running streaming statement.")



CREATE TABLE IF NOT EXISTS `orders_dup_stats_ahartono` (
  `window_start`       TIMESTAMP(3),
  `window_end`          TIMESTAMP(3),
  `total_records`      BIGINT,
  `unique_records`     BIGINT,
  `duplicate_records`  BIGINT,
  `duplicate_pct`      DOUBLE
) WITH (
  'changelog.mode' = 'append',
  'value.format'   = 'avro-registry'
);

✅ dup-stats sink ready.

INSERT INTO `orders_dup_stats_ahartono`
SELECT
  `window_start`,
  `window_end`,
  COUNT(*)                                  AS `total_records`,
  COUNT(DISTINCT `order_id`)                AS `unique_records`,
  COUNT(*) - COUNT(DISTINCT `order_id`)      AS `duplicate_records`,
  CAST(COUNT(*) - COUNT(DISTINCT `order_id`) AS DOUBLE) / COUNT(*) * 100 AS `duplicate_pct`
FROM TABLE(
  TUMBLE(TABLE `orders_source_ahartono`, DESCRIPTOR(`$rowtime`), INTERVAL '2' MINUTE)
)
GROUP BY `window_start`, `window_end`;

✅ Duplicate-rate stream submitted -- it stays RUNNING and keeps emitting.


## Step 3b -- Read Flink's duplicate-rate result back and compare to ground truth

The window closes 2 minutes after the last record's timestamp passes the watermark, so this may
need a couple of re-runs before a row shows up. Once it does, `duplicate_pct` should land close to
`GROUND_TRUTH_DUP_PCT` computed in Step 2 (not exact -- window boundaries can split the burst).

**Note:** this consumer reads `orders_dup_stats_<you>` from `earliest`, so every row ever written
to that sink -- including windows from earlier runs of this notebook -- comes back, not just the
latest one. The table below sorts by `window_end` and highlights the most recent window, since
that's the one that reflects the batch you just produced in Step 2.

In [63]:
stats_deser = AvroDeserializer(sr_client)

stats_consumer = Consumer({
    **kafka_conf,
    "group.id": f"dupstats-reader-{PARTICIPANT}-{uuid.uuid4().hex[:6]}",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
})
stats_consumer.subscribe([TOPIC_DUP_STATS])

# The window closes ~2 minutes after the last record's event-time passes the watermark,
# so give it enough time to show up rather than the few seconds a single poll pass takes.
# At higher source volumes (e.g. 20k rows from Step 2), producing + Flink's own processing
# of the backlog can itself take well over a minute, so both the overall deadline and the
# idle cutoff scale with EXPECTED_SOURCE_ROWS instead of being fixed constants tuned for
# a small burst.
WINDOW_SECS = 120  # matches the 2-minute tumbling window in Step 3's Flink SQL
EXPECTED_SOURCE_ROWS = globals().get("NUM_ORDERS_PRODUCED", 200)

# Baseline covers watermark advancement for one window; add headroom per extra
# 1,000 source rows since Flink needs proportionally longer to drain a bigger backlog
# before the watermark catches up and the window can close.
MAX_WAIT = max(150, WINDOW_SECS * 2 + (EXPECTED_SOURCE_ROWS // 1000) * 15)
IDLE_CUTOFF = max(20, (EXPECTED_SOURCE_ROWS // 1000) * 10)

print(f"{INFO} Waiting up to {MAX_WAIT}s total (idle cutoff {IDLE_CUTOFF}s) for "
      f"~{EXPECTED_SOURCE_ROWS} source rows to flow through Step 3's windows...")

rows, deadline = [], time.time() + MAX_WAIT
last_row_at, last_progress_print = None, time.time()
while time.time() < deadline:
    m = stats_consumer.poll(2.0)
    if m is None or m.error():
        if rows and last_row_at and time.time() - last_row_at > IDLE_CUTOFF:
            break
        # Heartbeat every 20s so a long wait on a big batch doesn't look hung.
        if time.time() - last_progress_print > 20:
            remaining = int(deadline - time.time())
            print(f"{INFO} Still waiting... {len(rows)} row(s) so far, {remaining}s left "
                  f"before deadline (idle cutoff resets on each new row).")
            last_progress_print = time.time()
        continue
    rows.append(stats_deser(m.value(), SerializationContext(TOPIC_DUP_STATS, MessageField.VALUE)))
    last_row_at = time.time()
stats_consumer.close()

if rows:
    df = pd.DataFrame(rows).sort_values("window_end").reset_index(drop=True)
    display(df)
    latest = df.iloc[-1]
    print(f"\n{INFO} Most recent window: total_records={latest['total_records']}, "
          f"duplicate_pct={latest['duplicate_pct']:.2f}%")
    print(f"{INFO} Ground truth duplicate %: {GROUND_TRUTH_DUP_PCT}% (from Step 2)")
    if latest['total_records'] < EXPECTED_SOURCE_ROWS:
        print(f"{WARN} Most recent window only covers {latest['total_records']} of "
              f"~{EXPECTED_SOURCE_ROWS} rows -- the batch likely spans multiple 2-minute "
              f"windows above; check the full table, not just the last row.")
else:
    print(f"{WARN} No rows yet after {MAX_WAIT}s. The window may not have closed -- wait a minute or two and re-run this cell.")
    print(f"     If it stays empty, confirm the Step 3 statement is RUNNING (Cloud Console -> Flink -> Statements).")

ℹ️ Waiting up to 240s total (idle cutoff 20s) for ~200 source rows to flow through Step 3's windows...


%6|1786892010.960|GETSUBSCRIPTIONS|rdkafka#consumer-36| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to SnM5B7OgTBGVmci4P2iWKA


ℹ️ Still waiting... 0 row(s) so far, 219s left before deadline (idle cutoff resets on each new row).
ℹ️ Still waiting... 0 row(s) so far, 199s left before deadline (idle cutoff resets on each new row).
ℹ️ Still waiting... 0 row(s) so far, 179s left before deadline (idle cutoff resets on each new row).
ℹ️ Still waiting... 0 row(s) so far, 158s left before deadline (idle cutoff resets on each new row).
ℹ️ Still waiting... 0 row(s) so far, 138s left before deadline (idle cutoff resets on each new row).
ℹ️ Still waiting... 0 row(s) so far, 118s left before deadline (idle cutoff resets on each new row).


%3|1786892146.430|FAIL|rdkafka#producer-24| [thrd:sasl_ssl://b7-pkc-921jm.us-east-2.aws.confluent.cloud:9092/7]: sasl_ssl://b7-pkc-921jm.us-east-2.aws.confluent.cloud:9092/7: Receive failed: SSL transport error: Operation timed out (after 225886ms in state UP)


ℹ️ Still waiting... 0 row(s) so far, 97s left before deadline (idle cutoff resets on each new row).
ℹ️ Still waiting... 0 row(s) so far, 77s left before deadline (idle cutoff resets on each new row).


%3|1786892189.718|FAIL|rdkafka#producer-24| [thrd:sasl_ssl://b21-pkc-921jm.us-east-2.aws.confluent.cloud:9092/21]: sasl_ssl://b21-pkc-921jm.us-east-2.aws.confluent.cloud:9092/21: Receive failed: SSL transport error: Operation timed out (after 568018ms in state UP)


ℹ️ Still waiting... 0 row(s) so far, 57s left before deadline (idle cutoff resets on each new row).
ℹ️ Still waiting... 0 row(s) so far, 36s left before deadline (idle cutoff resets on each new row).
ℹ️ Still waiting... 0 row(s) so far, 16s left before deadline (idle cutoff resets on each new row).
⚠️ No rows yet after 240s. The window may not have closed -- wait a minute or two and re-run this cell.
     If it stays empty, confirm the Step 3 statement is RUNNING (Cloud Console -> Flink -> Statements).


## Step 4 -- Deduplicate the stream into a clean topic

The classic Flink dedup pattern: number each row within its key using `ROW_NUMBER()` ordered by
event time, then keep only `row_num = 1`. This keeps the **first** occurrence of each `order_id`
and drops every resend.


In [64]:
DDL_DEDUPED = f"""
CREATE TABLE IF NOT EXISTS `{TOPIC_DEDUPED}` (
  `order_id`    STRING,
  `customer_id` STRING,
  `product_id`  STRING,
  `quantity`    INT,
  `amount`      DOUBLE,
  `order_ts`    TIMESTAMP(3),
  `status`      STRING,
  `channel`     STRING
) WITH (
  'changelog.mode' = 'append',
  'value.format'   = 'avro-registry'
);
"""

DEDUP_QUERY = f"""
INSERT INTO `{TOPIC_DEDUPED}`
SELECT `order_id`, `customer_id`, `product_id`, `quantity`, `amount`, `order_ts`, `status`, `channel`
FROM (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY `order_id`
      ORDER BY `$rowtime` ASC
    ) AS row_num
  FROM `{TOPIC_ORDERS}`
)
WHERE row_num = 1;
"""

print(DDL_DEDUPED)
ok, out = run_flink_sql(DDL_DEDUPED, "ddl-deduped")
print(f"{OK} deduped sink ready." if ok else f"{INFO} DDL not executed (see above).")

print(DEDUP_QUERY)
ok, out = run_flink_sql(DEDUP_QUERY, "dedup")
print(f"{OK} Dedup stream submitted -- {GROUND_TRUTH_UNIQUE} distinct orders should eventually land here." if ok
      else f"{INFO} Statement not submitted (see above).")



CREATE TABLE IF NOT EXISTS `orders_deduped_ahartono` (
  `order_id`    STRING,
  `customer_id` STRING,
  `product_id`  STRING,
  `quantity`    INT,
  `amount`      DOUBLE,
  `order_ts`    TIMESTAMP(3),
  `status`      STRING,
  `channel`     STRING
) WITH (
  'changelog.mode' = 'append',
  'value.format'   = 'avro-registry'
);

✅ deduped sink ready.

INSERT INTO `orders_deduped_ahartono`
SELECT `order_id`, `customer_id`, `product_id`, `quantity`, `amount`, `order_ts`, `status`, `channel`
FROM (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY `order_id`
      ORDER BY `$rowtime` ASC
    ) AS row_num
  FROM `orders_source_ahartono`
)
WHERE row_num = 1;



%3|1786892259.462|FAIL|rdkafka#producer-24| [thrd:sasl_ssl://b1-pkc-921jm.us-east-2.aws.confluent.cloud:9092/1]: sasl_ssl://b1-pkc-921jm.us-east-2.aws.confluent.cloud:9092/1: Receive failed: SSL transport error: Operation timed out (after 697912ms in state UP)


✅ Dedup stream submitted -- 20000 distinct orders should eventually land here.


## Step 5 -- Filter + trim fields into a new topic

A downstream analytics team only cares about **confirmed, high-value** orders, and has no use for
the internal `channel` routing field. This statement does both at once: it filters rows with a
`WHERE` clause and trims columns with an explicit `SELECT` list (versus `SELECT *`).


In [65]:
HIGH_VALUE_THRESHOLD = 500.0

DDL_FILTERED = f"""
CREATE TABLE IF NOT EXISTS `{TOPIC_FILTERED}` (
  `order_id`    STRING,
  `customer_id` STRING,
  `product_id`  STRING,
  `amount`      DOUBLE,
  `order_ts`    TIMESTAMP(3)
) WITH (
  'changelog.mode' = 'append',
  'value.format'   = 'avro-registry'
);
"""

FILTER_TRIM_QUERY = f"""
INSERT INTO `{TOPIC_FILTERED}`
SELECT `order_id`, `customer_id`, `product_id`, `amount`, `order_ts`
FROM `{TOPIC_ORDERS}`
WHERE `status` = 'CONFIRMED' AND `amount` > {HIGH_VALUE_THRESHOLD};
"""

print(DDL_FILTERED)
ok, out = run_flink_sql(DDL_FILTERED, "ddl-filtered")
print(f"{OK} filtered sink ready." if ok else f"{INFO} DDL not executed (see above).")

print(FILTER_TRIM_QUERY)
ok, out = run_flink_sql(FILTER_TRIM_QUERY, "filter-trim")
print(f"{OK} Filter+trim stream submitted. Note: `quantity`, `status`, and `channel` are gone -- "
      "by design, not by accident." if ok else f"{INFO} Statement not submitted (see above).")



CREATE TABLE IF NOT EXISTS `orders_high_value_ahartono` (
  `order_id`    STRING,
  `customer_id` STRING,
  `product_id`  STRING,
  `amount`      DOUBLE,
  `order_ts`    TIMESTAMP(3)
) WITH (
  'changelog.mode' = 'append',
  'value.format'   = 'avro-registry'
);

✅ filtered sink ready.

INSERT INTO `orders_high_value_ahartono`
SELECT `order_id`, `customer_id`, `product_id`, `amount`, `order_ts`
FROM `orders_source_ahartono`
WHERE `status` = 'CONFIRMED' AND `amount` > 500.0;

✅ Filter+trim stream submitted. Note: `quantity`, `status`, and `channel` are gone -- by design, not by accident.


## Step 6 -- Enrichment: join the stream against a product reference table

Real orders only carry a `product_id`; a case worker or analyst wants the product name, category,
and unit price alongside it. We produce a small, static product catalog to its own topic, then join
the order stream against it on `product_id` with a plain **`INNER JOIN`**, since this catalog
doesn't change during the demo.

> **Why `INNER JOIN`, not `LEFT JOIN`?** A `LEFT JOIN` between two unbounded streams can *retract*
> an already-emitted row: if an order arrives before its matching catalog row, Flink first emits a
> null-padded result, then retracts and re-emits it once the match shows up. That makes the
> operator's output an update/delete changelog, which our `changelog.mode = 'append'` sink can't
> consume -- Flink rejects the statement with `Table sink ... doesn't support consuming update and
> delete changes`. An `INNER JOIN` never emits a placeholder row to retract, so its output really is
> append-only and matches an append sink. The cost: an order whose `product_id` isn't in the catalog
> is silently dropped instead of appearing with null enrichment.

> **If your reference table *does* change over time** (a real slowly-changing dimension), the
> correct pattern is a **temporal join** (`FOR SYSTEM_TIME AS OF`) against a table declared with
> `PRIMARY KEY (...) ... WITH ('changelog.mode' = 'upsert')`. That always resolves to the version of
> the reference row that was current *when the order arrived*. The trade-off: declaring a primary
> key makes Flink split the topic into a separate Avro-record key (just the key columns) and a value
> schema with the key columns removed -- so the producer writing to it has to serialize key and
> value against those two distinct, Flink-generated schemas rather than one flat record. Worth
> knowing about, not needed for a catalog this static.


In [66]:
PRODUCT_CATALOG = [
    {"product_id": "SKU-001", "product_name": "Wireless Mouse",      "category": "Electronics", "unit_price": 25.99},
    {"product_id": "SKU-002", "product_name": "Mechanical Keyboard", "category": "Electronics", "unit_price": 89.50},
    {"product_id": "SKU-003", "product_name": "USB-C Hub",           "category": "Electronics", "unit_price": 45.00},
    {"product_id": "SKU-004", "product_name": "Office Chair",        "category": "Furniture",   "unit_price": 199.99},
    {"product_id": "SKU-005", "product_name": "Standing Desk",       "category": "Furniture",   "unit_price": 349.00},
    {"product_id": "SKU-006", "product_name": "Running Shoes",       "category": "Apparel",     "unit_price": 79.90},
    {"product_id": "SKU-007", "product_name": "Winter Jacket",       "category": "Apparel",     "unit_price": 129.00},
    {"product_id": "SKU-008", "product_name": "Blender",             "category": "Home",        "unit_price": 59.00},
    {"product_id": "SKU-009", "product_name": "Air Fryer",           "category": "Home",        "unit_price": 89.00},
    {"product_id": "SKU-010", "product_name": "Coffee Maker",        "category": "Home",        "unit_price": 65.00},
]

CATALOG_SCHEMA = {
    "type": "record",
    "name": "Product",
    "namespace": "com.bnpl.orders",
    "fields": [
        {"name": "product_id",   "type": "string"},
        {"name": "product_name", "type": "string"},
        {"name": "category",     "type": "string"},
        {"name": "unit_price",   "type": "double"},
    ],
}
catalog_subject = f"{TOPIC_CATALOG}-value"
sr_client.register_schema(catalog_subject, Schema(pyjson.dumps(CATALOG_SCHEMA), "AVRO"))
catalog_ser = AvroSerializer(sr_client, pyjson.dumps(CATALOG_SCHEMA))

catalog_producer = Producer(kafka_conf)
ok_count = 0
def on_delivery_catalog(err, msg):
    global ok_count
    if not err:
        ok_count += 1

for product in PRODUCT_CATALOG:
    ctx = SerializationContext(TOPIC_CATALOG, MessageField.VALUE)
    catalog_producer.produce(
        topic=TOPIC_CATALOG,
        key=key_ser(product["product_id"]),
        value=catalog_ser(product, ctx),
        on_delivery=on_delivery_catalog,
    )
catalog_producer.flush(30)
print(f"{OK} Published {ok_count}/{len(PRODUCT_CATALOG)} products to {TOPIC_CATALOG}")

DDL_ENRICHED = f"""
CREATE TABLE IF NOT EXISTS `{TOPIC_ENRICHED}` (
  `order_id`     STRING,
  `customer_id`  STRING,
  `product_id`   STRING,
  `product_name` STRING,
  `category`     STRING,
  `unit_price`   DOUBLE,
  `quantity`     INT,
  `amount`       DOUBLE,
  `order_ts`     TIMESTAMP(3)
) WITH (
  'changelog.mode' = 'append',
  'value.format'   = 'avro-registry'
);
"""

ENRICH_QUERY = f"""
INSERT INTO `{TOPIC_ENRICHED}`
SELECT
  o.`order_id`, o.`customer_id`, o.`product_id`,
  p.`product_name`, p.`category`, p.`unit_price`,
  o.`quantity`, o.`amount`, o.`order_ts`
FROM `{TOPIC_ORDERS}` AS o
JOIN `{TOPIC_CATALOG}` AS p
ON o.`product_id` = p.`product_id`;
"""

print(DDL_ENRICHED)
ok, out = run_flink_sql(DDL_ENRICHED, "ddl-enriched")
print(f"{OK} enriched sink ready." if ok else f"{INFO} DDL not executed (see above).")

print(ENRICH_QUERY)
ok, out = run_flink_sql(ENRICH_QUERY, "enrich")
print(f"{OK} Enrichment stream submitted." if ok else f"{INFO} Statement not submitted (see above).")


✅ Published 10/10 products to product_catalog_ahartono

CREATE TABLE IF NOT EXISTS `orders_enriched_ahartono` (
  `order_id`     STRING,
  `customer_id`  STRING,
  `product_id`   STRING,
  `product_name` STRING,
  `category`     STRING,
  `unit_price`   DOUBLE,
  `quantity`     INT,
  `amount`       DOUBLE,
  `order_ts`     TIMESTAMP(3)
) WITH (
  'changelog.mode' = 'append',
  'value.format'   = 'avro-registry'
);



%6|1786892277.259|GETSUBSCRIPTIONS|rdkafka#producer-37| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to xuQ4B3cyRWiGdzuCOkVWOA


✅ enriched sink ready.

INSERT INTO `orders_enriched_ahartono`
SELECT
  o.`order_id`, o.`customer_id`, o.`product_id`,
  p.`product_name`, p.`category`, p.`unit_price`,
  o.`quantity`, o.`amount`, o.`order_ts`
FROM `orders_source_ahartono` AS o
JOIN `product_catalog_ahartono` AS p
ON o.`product_id` = p.`product_id`;

✅ Enrichment stream submitted.


## Step 7 -- Read back the deduped, filtered, and enriched topics

Give the statements a minute or two to start producing, then run this cell (re-run if empty).


In [67]:
def peek_topic(topic, deserializer, limit=5, timeout_s=20):
    c = Consumer({
        **kafka_conf,
        "group.id": f"peek-{topic}-{uuid.uuid4().hex[:6]}",
        "auto.offset.reset": "earliest",
        "enable.auto.commit": False,
    })
    c.subscribe([topic])
    rows, deadline = [], time.time() + timeout_s
    while time.time() < deadline and len(rows) < limit:
        m = c.poll(2.0)
        if m is None or m.error():
            continue
        rows.append(deserializer(m.value(), SerializationContext(topic, MessageField.VALUE)))
    c.close()
    return rows

generic_deser = AvroDeserializer(sr_client)

for label, topic in [("Deduped", TOPIC_DEDUPED), ("Filtered + trimmed", TOPIC_FILTERED), ("Enriched", TOPIC_ENRICHED)]:
    rows = peek_topic(topic, generic_deser)
    print(f"\n--- {label}: {topic} ---")
    if rows:
        display(pd.DataFrame(rows))
    else:
        print(f"{WARN} No rows yet -- the Flink statement may still be starting. Re-run this cell shortly.")


%6|1786892291.923|GETSUBSCRIPTIONS|rdkafka#consumer-38| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to hOvgLonaRZOmgKaR2KK3EQ
%5|1786892292.971|REQTMOUT|rdkafka#producer-34| [thrd:sasl_ssl://b19-pkc-921jm.us-east-2.aws.confluent.cloud:9092/19]: sasl_ssl://b19-pkc-921jm.us-east-2.aws.confluent.cloud:9092/19: Timed out PushTelemetryRequest in flight (after 60095ms, timeout #0)
%4|1786892292.971|REQTMOUT|rdkafka#producer-34| [thrd:sasl_ssl://b19-pkc-921jm.us-east-2.aws.confluent.cloud:9092/19]: sasl_ssl://b19-pkc-921jm.us-east-2.aws.confluent.cloud:9092/19: Timed out 1 in-flight, 0 retry-queued, 0 out-queue, 0 partially-sent requests



--- Deduped: orders_deduped_ahartono ---
⚠️ No rows yet -- the Flink statement may still be starting. Re-run this cell shortly.


%6|1786892312.508|GETSUBSCRIPTIONS|rdkafka#consumer-39| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to 4wY2UjY6RXyS7SdjCIDc6Q



--- Filtered + trimmed: orders_high_value_ahartono ---


,order_id,customer_id,product_id,amount,order_ts
0,392ff4f9-8d63-4b32-97cc-7919c48aef41,CUST-0031,SKU-002,972.82,2026-08-16 14:53:11.557


%6|1786892334.130|GETSUBSCRIPTIONS|rdkafka#consumer-40| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to VYNhxb8KQMiYr/miElfsUw
%3|1786892334.569|FAIL|rdkafka#producer-35| [thrd:sasl_ssl://b20-pkc-921jm.us-east-2.aws.confluent.cloud:9092/20]: sasl_ssl://b20-pkc-921jm.us-east-2.aws.confluent.cloud:9092/20: Receive failed: SSL transport error: Operation timed out (after 341006ms in state UP)



--- Enriched: orders_enriched_ahartono ---


,order_id,customer_id,product_id,product_name,category,unit_price,quantity,amount,order_ts
0,6eb92f31-0b1d-42f4-8053-f5018863ee0c,CUST-0013,SKU-007,Winter Jacket,Apparel,129.00,1,1179.32,2026-08-16 14:53:11.475
1,26015c24-965e-4cf1-98c9-47b483646efb,CUST-0039,SKU-001,Wireless Mouse,Electronics,25.99,4,450.12,2026-08-16 14:53:11.475
2,14fb2fe9-14bf-404b-8b7d-5f36d5d2b54d,CUST-0013,SKU-010,Coffee Maker,Home,65.00,4,374.45,2026-08-16 14:53:11.475
3,f67b2ee1-e25a-4d79-a4bc-d38ebf1067ec,CUST-0003,SKU-008,Blender,Home,59.00,3,217.28,2026-08-16 14:53:11.475
4,f67b2ee1-e25a-4d79-a4bc-d38ebf1067ec,CUST-0003,SKU-008,Blender,Home,59.00,3,217.28,2026-08-16 14:53:11.475


### Smoke-test the Flink REST helper

Runs a trivial `SELECT 1;` through `run_flink_sql()` to confirm your Flink REST credentials actually work, without needing a real query or topic.

**Example:** prints `✅ Flink REST smoke test: COMPLETED: statement 'smoketest-...'` if the connection and auth are set up correctly; a `401`/`403` here means the `FLINK_API_KEY`/`FLINK_API_SECRET` in `.env` are wrong.

In [68]:
# Quick smoke test of the Flink REST helper -- safe to delete once you've confirmed it works.
ok, out = run_flink_sql("SELECT 1;", "smoketest", wait_seconds=30)
print(f"{OK if ok else FAIL} Flink REST smoke test: {out}")


✅ Flink REST smoke test: RUNNING: statement 'smoketest-ahartono-cd2cc2'


## Cleanup

Set `CLEANUP = True` and re-run this cell to tear down everything this notebook created: the six
topics, their schema subjects, and any Flink statements tagged with your participant namespace.


In [69]:
CLEANUP = False

if not CLEANUP:
    print(f"{INFO} Cleanup disabled. Set CLEANUP = True and re-run to tear down:")
    print(f"     topics   : {TOPIC_ORDERS}, {TOPIC_DUP_STATS}, {TOPIC_DEDUPED}, {TOPIC_FILTERED}, {TOPIC_CATALOG}, {TOPIC_ENRICHED}")
    print(f"     subjects : {TOPIC_ORDERS}-value, {TOPIC_CATALOG}-value")
    print(f"     Flink    : all statements matching '{PARTICIPANT}'")
else:
    if FLINK_REST_AVAILABLE:
        auth = (FLINK_API_KEY, FLINK_API_SECRET)
        try:
            r = requests.get(FLINK_REST_BASE, auth=auth, timeout=30)
            r.raise_for_status()
            names = [s["name"] for s in r.json().get("data", [])
                     if PARTICIPANT.lower() in str(s.get("name", "")).lower()]
            for n in names:
                rr = requests.delete(f"{FLINK_REST_BASE}/{n}", auth=auth, timeout=30)
                print(f"{OK if rr.status_code in (200, 202, 204) else FAIL} statement {n}")
            if not names:
                print(f"{INFO} No matching Flink statements.")
        except Exception as e:
            print(f"{WARN} Flink cleanup: {e}")

    for subj in (f"{TOPIC_ORDERS}-value", f"{TOPIC_CATALOG}-value"):
        for permanent in (False, True):
            try:
                sr_client.delete_subject(subj, permanent=permanent)
                print(f"{OK} subject {subj} ({'hard' if permanent else 'soft'} delete)")
            except Exception as e:
                print(f"{INFO} subject {subj} ({'hard' if permanent else 'soft'}): {str(e)[:90]}")

    topics_to_delete = [TOPIC_ORDERS, TOPIC_DUP_STATS, TOPIC_DEDUPED, TOPIC_FILTERED, TOPIC_CATALOG, TOPIC_ENRICHED]
    for name, fut in admin.delete_topics(topics_to_delete).items():
        try:
            fut.result(timeout=60)
            print(f"{OK} topic {name} deleted")
        except Exception as e:
            print(f"{INFO} topic {name}: {str(e)[:90]}")

    print(f"\n{OK} Cleanup complete.")


ℹ️ Cleanup disabled. Set CLEANUP = True and re-run to tear down:
     topics   : orders_source_ahartono, orders_dup_stats_ahartono, orders_deduped_ahartono, orders_high_value_ahartono, product_catalog_ahartono, orders_enriched_ahartono
     subjects : orders_source_ahartono-value, product_catalog_ahartono-value
     Flink    : all statements matching 'ahartono'


%5|1786892395.041|REQTMOUT|rdkafka#producer-35| [thrd:sasl_ssl://b11-pkc-921jm.us-east-2.aws.confluent.cloud:9092/11]: sasl_ssl://b11-pkc-921jm.us-east-2.aws.confluent.cloud:9092/11: Timed out MetadataRequest in flight (after 60471ms, timeout #0)
%5|1786892395.041|REQTMOUT|rdkafka#producer-35| [thrd:sasl_ssl://b11-pkc-921jm.us-east-2.aws.confluent.cloud:9092/11]: sasl_ssl://b11-pkc-921jm.us-east-2.aws.confluent.cloud:9092/11: Timed out MetadataRequest in flight (after 60042ms, timeout #1)
%4|1786892395.041|REQTMOUT|rdkafka#producer-35| [thrd:sasl_ssl://b11-pkc-921jm.us-east-2.aws.confluent.cloud:9092/11]: sasl_ssl://b11-pkc-921jm.us-east-2.aws.confluent.cloud:9092/11: Timed out 2 in-flight, 0 retry-queued, 0 out-queue, 0 partially-sent requests
